# Retrain the 3 weaker models with more epochs

Every model was originally trained for **only 10 epochs** (found by reading the metadata baked into each `.pt` file - see `models/*.pt`). That's low for a from-pretrained fine-tune; typical runs are 50-300+ epochs. Before assuming rip_current, swimmer_multiclass, and swimmer_distress have a real data/architecture problem, this reruns each on the *exact same dataset version* they were originally trained on, just with epochs bumped 10 -> 100, to rule out "just undertrained."

Run this in Google Colab with a GPU runtime (Runtime -> Change runtime type -> GPU).

Needs your `ROBOFLOW_API_KEY` set as a Colab secret (key icon in the left sidebar) - same one referenced in the project handoff doc.

**Shark is not included** - its training-run metrics (mAP50 0.94) are already strong, no reason to spend compute re-running it.

In [ ]:
!pip install -q ultralytics roboflow

from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO

api_key = userdata.get("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)

EPOCHS = 100      # was 10 in the original training run
IMGSZ = 640       # unchanged from the original run
BATCH = 16        # unchanged from the original run
BASE_MODEL = "yolo11s.pt"  # same pretrained base as the original run

## 1. Rip current

Original training metrics: precision 0.561, recall 0.553, mAP50 0.561, mAP50-95 0.222 - the weakest of the four.

In [ ]:
project = rf.workspace("logan-friedman").project("rip-rm14s-dud0p")
version = project.version(1)  # version 1 - matches the folder name ("Rip-1") the original model was trained on
dataset = version.download("yolov8")

model = YOLO(BASE_MODEL)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, name="rip_retrain")

# best.pt for this run: runs/detect/rip_retrain/weights/best.pt

## 2. Swimmer multiclass (swimmer / boat / jetski / buoy / life_saving_appliances)

Original training metrics: precision 0.714, recall 0.568, mAP50 0.574, mAP50-95 0.343.

In [ ]:
project = rf.workspace("sea-drone").project("seadronessee-odv2-rebalancedtestset-n60cw")
version = project.version(3)  # version 3 - matches "SeaDronesSee-ODV2-REBALANCEDTESTSET-3", the folder the original model was trained on
dataset = version.download("yolov8")

model = YOLO(BASE_MODEL)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, name="swimmer_multiclass_retrain")

# best.pt for this run: runs/detect/swimmer_multiclass_retrain/weights/best.pt

## 3. Swimmer distress (drowning / swimming)

Original training metrics: precision 0.633, recall 0.679, mAP50 0.695, mAP50-95 0.364.

**Confirm before running:** the handoff doc lists two possible source datasets for this model and wasn't sure which one was actually used - `liptwo-z7hes/drowning-detection-qsuzf` or `zidan-nlsjs/drowning-detection-e6kbk`. The cell below defaults to the first; if the download's class names or image count look unfamiliar, try the second workspace/project instead.

In [ ]:
project = rf.workspace("liptwo-z7hes").project("drowning-detection-qsuzf")
# if this isn't the right source, comment the line above and use instead:
# project = rf.workspace("zidan-nlsjs").project("drowning-detection-e6kbk")

version = project.version(7)  # version 7 - matches "Drowning-Detection-7", the folder the original model was trained on
dataset = version.download("yolov8")

model = YOLO(BASE_MODEL)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, name="swimmer_distress_retrain")

# best.pt for this run: runs/detect/swimmer_distress_retrain/weights/best.pt

## Next steps

1. Download each `best.pt` from `runs/detect/<name>_retrain/weights/best.pt` (Colab sessions are temporary - save these somewhere permanent, e.g. mount Google Drive and copy them there, before the runtime disconnects).
2. Send the new `.pt` files back so they can be swapped into `models/` and re-checked - I'll read the same training-metadata trick used to find the original weak scores, so we immediately know whether more epochs actually helped before touching anything else.
3. This still isn't a substitute for a genuine held-out test set (Step 0 in the validation guide) - it only tells us whether "more epochs" fixes it, not the full picture.